In [6]:
import requests
import csv
import pandas as pd

headers = {
    'Content-Type': 'application/json',
    'Accept': 'application/json',
}

input_file = 'contracts_search_results  2025 SERV.xlsx'
output_file = 'simbaseis 2025 SERV.csv'

# Read Excel file
df = pd.read_excel(input_file)

# Open output CSV once
with open(output_file, 'w', newline='', encoding='iso-8859-7') as csvfile:

    writer = csv.writer(csvfile)

    # Header row
    writer.writerow([
        'referenceNumber',
        'vatNumber',
        'totalCostWithVAT',
        'totalCostWithoutVAT',
        'cpvs',
        'signers',
        'contractingMembersDataList',
        'contractRefNo',
        'paymentRefNo',
        'contractType',
        'procedureType',
        'signedDate',
        'contractTitle'
    ])

    # Loop through Excel rows
    for index, row in df.iterrows():

        reference_number = row['Κωδικός ΑΔΑΜ']

        # Skip empty cells
        if pd.isna(reference_number):
            continue

        reference_number = str(reference_number).strip()

        json_data = {
            'referenceNumber': reference_number
        }

        try:

            response = requests.post(
                #'https://cerpp.eprocurement.gov.gr/khmdhs-opendata/auction',
                'https://cerpp.eprocurement.gov.gr/khmdhs-opendata/contract',
                params={'page': '0'},
                headers=headers,
                json=json_data,
            )

            data = response.json()
            #print(data)

            def clean_unicode(obj):
              if isinstance(obj, dict):
                  return {k: clean_unicode(v) for k, v in obj.items()}
              elif isinstance(obj, list):
                  return [clean_unicode(v) for v in obj]
              elif isinstance(obj, str):
                  return (
                      obj.replace('\u2013', '-')
                        .replace('\u201c', '"')
                        .replace('\u201d', '"')
                  )
              return obj

            cleaned_data = clean_unicode(data)
            data = cleaned_data

            # Skip empty API results
            if not data.get('content'):
                print(f'No results for {reference_number}')
                continue

            contract = data['content'][0]
            contract_ref_no = contract.get('contractRefNo', [])

            contract_ref_no = ' '.join(contract_ref_no)

            print(contract_ref_no)

            if contract_ref_no != '':
                print(f'Simbasi')
                continue

            ifcancelled = contract.get('cancelled')
            if ifcancelled == True:
                print(f'Cancelled')
                continue

            # VAT number
            vat_number = None

            members = contract.get(
                'contractingDataDetails',
                {}
            ).get(
                'contractingMembersDataList',
                []
            )

            if members:
                vat_number = members[0].get('vatNumber')

            # Costs

            #print(contract)

            total_with_vat = contract.get('totalCostWithVAT')
            total_without_vat = contract.get('totalCostWithoutVAT')


            contractSignedDate = None
            contractSignedDate = contract.get('contractSignedDate')
            #print(contractSignedDate)

            contractTitle = None
            contractTitle = contract.get('title')
            #print(contractTitle)

            # Extract CPV codes
            cpvs = []

            for obj in contract.get('objectDetailsList', []):

                for cpv in obj.get('cpvs', []):

                    cpv_code = cpv.get('key')

                    if cpv_code:
                        cpvs.append(cpv_code)

            cpvs_string = ', '.join(cpvs)

            # Signers
            signers = contract.get(
                'contractingDataDetails',
                {}
            ).get(
                'signers',
                {}
            ).get('value')
            # Contracting Members Names
            member_names = []

            for member in members:

                name = member.get('name')

                if name:
                    member_names.append(name)

            members_string = ', '.join(member_names)

            # Payment References

            payment_refs = contract.get('paymentRefNo', [])

            payment_refs_string = ', '.join(payment_refs)

            # Contract Type
            contract_type = contract.get(
                'contractType',
                {}
            ).get('value')

            # Procedure Type
            procedure_type = contract.get(
                'procedureType',
                {}
            )

            if isinstance(procedure_type, dict):
                procedure_type = procedure_type.get('value')

            # Write row
            writer.writerow([
                reference_number,
                vat_number,
                total_with_vat,
                total_without_vat,
                cpvs_string,
                signers,
                members_string,
                contract_ref_no,
                payment_refs_string,
                contract_type,
                procedure_type,
                contractSignedDate,
                contractTitle
            ])


            print(f"Processed: {reference_number}")

        except Exception as e:

            print(f"Error with {reference_number}: {e}")

print("CSV file saved successfully.")

/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")



Processed: 25SYMV018287530

Processed: 25SYMV018286355

Processed: 25SYMV018283413

Processed: 25SYMV018283309

Processed: 25SYMV018283159

Processed: 25SYMV018283089

Processed: 25SYMV018282707

Processed: 25SYMV018281889

Processed: 25SYMV018281208

Processed: 25SYMV018277431

Processed: 25SYMV018277379

Cancelled

Processed: 25SYMV018266784

Processed: 25SYMV018250244

Processed: 25SYMV018245870

Processed: 25SYMV018229457

Processed: 25SYMV018221497

Processed: 25SYMV018199364

Processed: 25SYMV018177144

Processed: 25SYMV018175777

Processed: 25SYMV018168880

Processed: 25SYMV018164197

Processed: 25SYMV018070261

Processed: 25SYMV018061383

Processed: 25SYMV018047489

Processed: 25SYMV018015335

Processed: 25SYMV018013377

Processed: 25SYMV018010578

Processed: 25SYMV017953947

Processed: 25SYMV017947876

Processed: 25SYMV017931743

Processed: 25SYMV017878803

Processed: 25SYMV017866310

Processed: 25SYMV017857987

Processed: 25SYMV017845335

Processed: 25SYMV017843859

Processe